# Análise e Visualização dos Dados Xavier Daily

Este notebook contém análises e visualizações dos dados de precipitação diária do Brasil (Xavier et al.) armazenados em arquivos NetCDF.

**Arquivos de dados:**
- `pr_19610101_19801231_BR-DWGD_UFES_UTEXAS_v_3.2.2-002.nc` (1961-1980)
- `pr_19810101_20001231_BR-DWGD_UFES_UTEXAS_v_3.2.2-003.nc` (1981-2000)
- `pr_20010101_20191231_BR-DWGD_UFES_UTEXAS_v_3.2.2-001.nc` (2001-2019)
- `final_2020_ate_2024-004.nc` (2020-2024)

**Fonte:** Xavier, A.C., King, C.W., Scanlon, B.R. Daily gridded meteorological variables in Brazil (1980-2013). International Journal of Climatology, 2016.

## 1. Import Required Libraries

In [3]:
# Bibliotecas para manipulação de dados
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import os

# Bibliotecas para visualização
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Configurações de visualização
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Suprimir avisos
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## 2. Load Data from xavier_daily

In [ ]:
# Definir caminhos dos dados
data_path = Path("../data/raw/xavier_daily/")
output_path = Path("../output/")

# Lista de arquivos NetCDF disponíveis
nc_files = {
    "1961-1980": "pr_19610101_19801231_BR-DWGD_UFES_UTEXAS_v_3.2.2-002.nc",
    "1981-2000": "pr_19810101_20001231_BR-DWGD_UFES_UTEXAS_v_3.2.2-003.nc", 
    "2001-2019": "pr_20010101_20191231_BR-DWGD_UFES_UTEXAS_v_3.2.2-001.nc",
    "2020-2024": "final_2020_ate_2024-004.nc"
}

# Função para carregar um arquivo NetCDF
def load_netcdf_file(file_path):
    """Carrega um arquivo NetCDF e retorna o dataset xarray"""
    try:
        ds = xr.open_dataset(file_path)
        print(f"Arquivo carregado: {file_path.name}")
        print(f"Dimensões: {dict(ds.dims)}")
        print(f"Variáveis: {list(ds.data_vars)}")
        print(f"Período: {ds.time.min().values} a {ds.time.max().values}")
        print("-" * 50)
        return ds
    except Exception as e:
        print(f"Erro ao carregar {file_path}: {e}")
        return None

# Carregar todos os arquivos disponíveis
datasets = {}
for period, filename in nc_files.items():
    file_path = data_path / filename
    if file_path.exists():
        datasets[period] = load_netcdf_file(file_path)
    else:
        print(f"Arquivo não encontrado: {filename}")

print(f"\nTotal de datasets carregados: {len(datasets)}")
print(f"Períodos disponíveis: {list(datasets.keys())}")

Arquivo carregado: pr_19610101_19801231_BR-DWGD_UFES_UTEXAS_v_3.2.2-002.nc
Dimensões: {'time': 7305, 'latitude': 393, 'longitude': 391}
Variáveis: ['pr']
Período: 1961-01-01T00:00:00.000000000 a 1980-12-31T00:00:00.000000000
--------------------------------------------------
Arquivo carregado: pr_19810101_20001231_BR-DWGD_UFES_UTEXAS_v_3.2.2-003.nc
Dimensões: {'time': 7305, 'latitude': 393, 'longitude': 391}
Variáveis: ['pr']
Período: 1981-01-01T00:00:00.000000000 a 2000-12-31T00:00:00.000000000
--------------------------------------------------
Arquivo carregado: pr_20010101_20191231_BR-DWGD_UFES_UTEXAS_v_3.2.2-001.nc
Dimensões: {'time': 6939, 'latitude': 393, 'longitude': 391}
Variáveis: ['pr']
Período: 2001-01-01T00:00:00.000000000 a 2019-12-31T00:00:00.000000000
--------------------------------------------------
Arquivo carregado: final_2020_ate_2024-004.nc
Dimensões: {'time': 1827, 'latitude': 393, 'longitude': 391}
Variáveis: ['ETo', 'RH', 'Rs', 'Tmax', 'Tmin', 'pr', 'u2', 'inter

: 

### 2.1 Processar Dados para Acumulado Mensal

Antes de continuar com a análise, vamos processar os dados diários para acumulados mensais para reduzir o volume de dados e facilitar as análises.

In [ ]:
# Importar o módulo de processamento
import sys
sys.path.append('../src')

from data_processing.transform.xavier_pr_accumulated import XavierPrecipitationProcessor

# Função para processar dados em formato mensal (executar apenas uma vez)
def process_to_monthly():
    """Processar dados diários para acumulados mensais."""
    
    # Definir caminhos
    input_path = "../data/raw/xavier_daily/"
    output_path = "../data/processed/xavier_monthly/"
    
    print("🔄 Iniciando processamento para dados mensais...")
    print(f"Entrada: {input_path}")
    print(f"Saída: {output_path}")
    
    # Criar processador
    processor = XavierPrecipitationProcessor(
        input_path=input_path,
        output_path=output_path,
        chunk_size="100MB"  # Ajustar conforme memória disponível
    )
    
    # Verificar arquivos disponíveis
    existing_files = processor.check_input_files()
    
    # Perguntar se deve processar
    if any(existing_files.values()):
        proceed = input("\n🤔 Deseja processar os dados para formato mensal? (y/n): ").lower().strip() == 'y'
        
        if proceed:
            print("\n📊 Processando dados...")
            results = processor.process_all_files()
            
            # Criar relatório
            report_path = processor.create_validation_report(results)
            
            success_count = sum(1 for path in results.values() if path is not None)
            print(f"\n✅ Processamento concluído!")
            print(f"   - Arquivos processados: {success_count}/{len(results)}")
            print(f"   - Relatório: {report_path}")
            
            return results
        else:
            print("⏭️ Processamento cancelado. Usando dados diários...")
            return None
    else:
        print("❌ Nenhum arquivo de entrada encontrado!")
        return None

# Verificar se dados mensais já existem
monthly_path = Path("../data/processed/xavier_monthly/")
monthly_files = list(monthly_path.glob("xavier_pr_monthly_*.nc")) if monthly_path.exists() else []

if monthly_files:
    print(f"✅ Dados mensais já existem ({len(monthly_files)} arquivos)")
    print("Arquivos encontrados:")
    for file in monthly_files:
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  - {file.name} ({size_mb:.1f} MB)")
    
    use_monthly = input("\n🤔 Usar dados mensais existentes para análise? (y/n): ").lower().strip() == 'y'
    
    if use_monthly:
        print("📊 Carregando dados mensais...")
        # Atualizar caminhos para usar dados mensais
        data_path = Path("../data/processed/xavier_monthly/")
        
        # Definir arquivos mensais
        nc_files = {}
        for file in monthly_files:
            # Extrair período do nome do arquivo
            period_part = file.stem.replace("xavier_pr_monthly_", "").replace("_", "-")
            nc_files[period_part] = file.name
        
        print(f"Dados mensais carregados: {list(nc_files.keys())}")
    else:
        print("📈 Usando dados diários originais...")
else:
    print("📋 Dados mensais não encontrados.")
    process_results = process_to_monthly()
    
    if process_results:
        # Atualizar para usar dados mensais se processamento foi bem-sucedido
        data_path = Path("../data/processed/xavier_monthly/")
        monthly_files = list(data_path.glob("xavier_pr_monthly_*.nc"))
        
        nc_files = {}
        for file in monthly_files:
            period_part = file.stem.replace("xavier_pr_monthly_", "").replace("_", "-")
            nc_files[period_part] = file.name

## 3. Data Exploration and Overview

In [ ]:
# Explorar estrutura dos dados para cada período
for period, ds in datasets.items():
    if ds is not None:
        print(f"\n=== ANÁLISE DO PERÍODO {period} ===")
        print(f"Forma dos dados: {ds.dims}")
        
        # Informações sobre coordenadas
        print(f"\nCoordenadas:")
        for coord in ds.coords:
            coord_data = ds.coords[coord]
            if coord == 'time':
                print(f"  {coord}: {len(coord_data)} timesteps ({coord_data.min().values} - {coord_data.max().values})")
            else:
                print(f"  {coord}: {len(coord_data)} pontos ({coord_data.min().values:.2f} - {coord_data.max().values:.2f})")
        
        # Informações sobre variáveis
        print(f"\nVariáveis de dados:")
        for var in ds.data_vars:
            var_data = ds[var]
            print(f"  {var}: {var_data.shape} - {var_data.attrs.get('long_name', 'Sem descrição')}")
            print(f"    Unidades: {var_data.attrs.get('units', 'Não especificado')}")
            print(f"    Min: {var_data.min().values:.2f}, Max: {var_data.max().values:.2f}")
            print(f"    Valor médio: {var_data.mean().values:.2f}")
        
        # Verificar valores ausentes
        if 'pr' in ds.data_vars:
            pr_data = ds['pr']
            missing_values = pr_data.isnull().sum().values
            total_values = pr_data.size
            print(f"\nValores ausentes: {missing_values:,} ({missing_values/total_values*100:.2f}%)")
            
        print("=" * 60)


=== ANÁLISE DO PERÍODO 1961-1980 ===
Forma dos dados: Frozen({'time': 7305, 'latitude': 393, 'longitude': 391})

Coordenadas:
  time: 7305 timesteps (1961-01-01T00:00:00.000000000 - 1980-12-31T00:00:00.000000000)
  latitude: 393 pontos (-33.85 - 5.35)
  longitude: 391 pontos (-73.85 - -34.85)

Variáveis de dados:
  pr: (7305, 393, 391) - Sem descrição
    Unidades: mm


## 4. Basic Data Visualization

In [ ]:
# Visualizações básicas para entender distribuição dos dados
def plot_basic_stats(datasets):
    """Criar visualizações básicas dos dados de precipitação"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Análise Básica dos Dados Xavier Daily', fontsize=16, fontweight='bold')
    
    # Coletar estatísticas de todos os períodos
    all_data = []
    periods = []
    
    for period, ds in datasets.items():
        if ds is not None and 'pr' in ds.data_vars:
            # Dados de precipitação (converter para DataFrame para facilitar a análise)
            pr_data = ds['pr'].values.flatten()
            # Remover valores NaN
            pr_data = pr_data[~np.isnan(pr_data)]
            all_data.extend(pr_data)
            periods.extend([period] * len(pr_data))
    
    if not all_data:
        print("Nenhum dado de precipitação encontrado!")
        return
    
    # 1. Histograma geral da precipitação
    axes[0,0].hist(all_data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0,0].set_xlabel('Precipitação (mm)')
    axes[0,0].set_ylabel('Frequência')
    axes[0,0].set_title('Distribuição da Precipitação Diária')
    axes[0,0].set_xlim(0, np.percentile(all_data, 99))  # Limitar aos 99% para melhor visualização
    
    # 2. Box plot por período
    data_by_period = []
    period_labels = []
    for period, ds in datasets.items():
        if ds is not None and 'pr' in ds.data_vars:
            pr_data = ds['pr'].values.flatten()
            pr_data = pr_data[~np.isnan(pr_data)]
            data_by_period.append(pr_data)
            period_labels.append(period)
    
    if data_by_period:
        axes[0,1].boxplot(data_by_period, labels=period_labels, patch_artist=True)
        axes[0,1].set_ylabel('Precipitação (mm)')
        axes[0,1].set_title('Distribuição por Período')
        axes[0,1].set_ylim(0, np.percentile(all_data, 95))
        plt.setp(axes[0,1].get_xticklabels(), rotation=45)
    
    # 3. Estatísticas descritivas por período
    stats_data = []
    for period, ds in datasets.items():
        if ds is not None and 'pr' in ds.data_vars:
            pr_data = ds['pr'].values.flatten()
            pr_data = pr_data[~np.isnan(pr_data)]
            stats_data.append([
                np.mean(pr_data),
                np.median(pr_data), 
                np.std(pr_data),
                np.percentile(pr_data, 95)
            ])
    
    if stats_data:
        stats_df = pd.DataFrame(stats_data, 
                               columns=['Média', 'Mediana', 'Desvio Padrão', 'P95'],
                               index=period_labels)
        
        x_pos = np.arange(len(period_labels))
        width = 0.2
        
        axes[1,0].bar(x_pos - width*1.5, stats_df['Média'], width, label='Média', alpha=0.8)
        axes[1,0].bar(x_pos - width*0.5, stats_df['Mediana'], width, label='Mediana', alpha=0.8)
        axes[1,0].bar(x_pos + width*0.5, stats_df['Desvio Padrão'], width, label='Desvio Padrão', alpha=0.8)
        axes[1,0].bar(x_pos + width*1.5, stats_df['P95'], width, label='Percentil 95', alpha=0.8)
        
        axes[1,0].set_xlabel('Período')
        axes[1,0].set_ylabel('Precipitação (mm)')
        axes[1,0].set_title('Estatísticas Descritivas por Período')
        axes[1,0].set_xticks(x_pos)
        axes[1,0].set_xticklabels(period_labels, rotation=45)
        axes[1,0].legend()
    
    # 4. Distribuição de dias secos vs chuvosos
    dry_wet_stats = []
    for period, ds in datasets.items():
        if ds is not None and 'pr' in ds.data_vars:
            pr_data = ds['pr'].values.flatten()
            pr_data = pr_data[~np.isnan(pr_data)]
            
            dry_days = np.sum(pr_data < 1.0)  # Menos de 1mm considerado dia seco
            wet_days = np.sum(pr_data >= 1.0)
            total_days = len(pr_data)
            
            dry_wet_stats.append([
                dry_days/total_days*100,
                wet_days/total_days*100
            ])
    
    if dry_wet_stats:
        dry_wet_df = pd.DataFrame(dry_wet_stats,
                                columns=['Dias Secos (%)', 'Dias Chuvosos (%)'],
                                index=period_labels)
        
        dry_wet_df.plot(kind='bar', ax=axes[1,1], alpha=0.8)
        axes[1,1].set_xlabel('Período')
        axes[1,1].set_ylabel('Percentual de Dias (%)')
        axes[1,1].set_title('Distribuição de Dias Secos vs Chuvosos')
        axes[1,1].legend()
        axes[1,1].set_xticklabels(period_labels, rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return stats_df if 'stats_df' in locals() else None

# Executar análise básica
stats_summary = plot_basic_stats(datasets)
if stats_summary is not None:
    print("\nResumo das Estatísticas:")
    print(stats_summary.round(2))

## 5. Time Series Plotting

In [ ]:
# Análise temporal dos dados de precipitação
def plot_time_series_analysis(datasets):
    """Criar visualizações de séries temporais"""
    
    fig, axes = plt.subplots(3, 2, figsize=(18, 15))
    fig.suptitle('Análise Temporal dos Dados de Precipitação', fontsize=16, fontweight='bold')
    
    # Combinar todos os datasets para análise temporal completa
    combined_data = []
    combined_times = []
    
    for period, ds in sorted(datasets.items()):
        if ds is not None and 'pr' in ds.data_vars:
            # Calcular média espacial diária
            daily_mean = ds['pr'].mean(dim=['lat', 'lon'])
            combined_data.extend(daily_mean.values)
            combined_times.extend(pd.to_datetime(ds.time.values))
    
    if not combined_data:
        print("Nenhum dado temporal encontrado!")
        return
    
    # Criar série temporal combinada
    ts_df = pd.DataFrame({
        'date': combined_times,
        'precipitation': combined_data
    })
    ts_df = ts_df.sort_values('date').reset_index(drop=True)
    ts_df = ts_df.dropna()
    
    # 1. Série temporal completa (média móvel de 30 dias)
    ts_df['rolling_30d'] = ts_df['precipitation'].rolling(window=30, center=True).mean()
    
    axes[0,0].plot(ts_df['date'], ts_df['precipitation'], alpha=0.3, color='lightblue', linewidth=0.5)
    axes[0,0].plot(ts_df['date'], ts_df['rolling_30d'], color='navy', linewidth=2, label='Média móvel 30d')
    axes[0,0].set_xlabel('Data')
    axes[0,0].set_ylabel('Precipitação (mm)')
    axes[0,0].set_title('Série Temporal Completa - Média Nacional Diária')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Sazonalidade - Ciclo anual médio
    ts_df['month'] = ts_df['date'].dt.month
    monthly_avg = ts_df.groupby('month')['precipitation'].agg(['mean', 'std']).reset_index()
    
    axes[0,1].errorbar(monthly_avg['month'], monthly_avg['mean'], 
                      yerr=monthly_avg['std'], capsize=5, capthick=2, alpha=0.8)
    axes[0,1].plot(monthly_avg['month'], monthly_avg['mean'], 'o-', linewidth=2, markersize=8)
    axes[0,1].set_xlabel('Mês')
    axes[0,1].set_ylabel('Precipitação (mm)')
    axes[0,1].set_title('Ciclo Sazonal Médio')
    axes[0,1].set_xticks(range(1, 13))
    axes[0,1].set_xticklabels(['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                              'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Tendências anuais
    ts_df['year'] = ts_df['date'].dt.year
    yearly_stats = ts_df.groupby('year')['precipitation'].agg(['mean', 'sum', 'std']).reset_index()
    
    # Linha de tendência
    z = np.polyfit(yearly_stats['year'], yearly_stats['mean'], 1)
    p = np.poly1d(z)
    
    axes[1,0].scatter(yearly_stats['year'], yearly_stats['mean'], alpha=0.7, s=50)
    axes[1,0].plot(yearly_stats['year'], p(yearly_stats['year']), 'r--', linewidth=2, 
                   label=f'Tendência: {z[0]:.3f} mm/ano')
    axes[1,0].set_xlabel('Ano')
    axes[1,0].set_ylabel('Precipitação Média Anual (mm/dia)')
    axes[1,0].set_title('Tendência da Precipitação Anual')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Variabilidade interanual
    axes[1,1].fill_between(yearly_stats['year'], 
                          yearly_stats['mean'] - yearly_stats['std'],
                          yearly_stats['mean'] + yearly_stats['std'],
                          alpha=0.3, label='±1 Desvio Padrão')
    axes[1,1].plot(yearly_stats['year'], yearly_stats['mean'], 'o-', linewidth=2, markersize=6)
    axes[1,1].set_xlabel('Ano')
    axes[1,1].set_ylabel('Precipitação (mm/dia)')
    axes[1,1].set_title('Variabilidade Interanual')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # 5. Análise de extremos por década
    ts_df['decade'] = (ts_df['year'] // 10) * 10
    decade_extremes = ts_df.groupby('decade')['precipitation'].agg(['min', 'max', 'quantile']).round(2)
    
    decades = decade_extremes.index
    axes[2,0].bar(decades - 2, decade_extremes['max'], width=4, alpha=0.7, 
                 label='Máximo', color='red')
    axes[2,0].bar(decades + 2, decade_extremes['min'], width=4, alpha=0.7, 
                 label='Mínimo', color='blue')
    axes[2,0].set_xlabel('Década')
    axes[2,0].set_ylabel('Precipitação (mm)')
    axes[2,0].set_title('Extremos de Precipitação por Década')
    axes[2,0].legend()
    axes[2,0].grid(True, alpha=0.3)
    
    # 6. Distribuição de precipitação por estação do ano
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Verão'
        elif month in [3, 4, 5]:
            return 'Outono'
        elif month in [6, 7, 8]:
            return 'Inverno'
        else:
            return 'Primavera'
    
    ts_df['season'] = ts_df['month'].apply(get_season)
    seasonal_data = [ts_df[ts_df['season'] == season]['precipitation'].values 
                    for season in ['Verão', 'Outono', 'Inverno', 'Primavera']]
    
    bp = axes[2,1].boxplot(seasonal_data, labels=['Verão', 'Outono', 'Inverno', 'Primavera'],
                          patch_artist=True)
    
    colors = ['orange', 'brown', 'lightblue', 'green']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    axes[2,1].set_ylabel('Precipitação (mm)')
    axes[2,1].set_title('Distribuição Sazonal da Precipitação')
    axes[2,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return ts_df, yearly_stats

# Executar análise temporal
print("Executando análise de séries temporais...")
time_series_data, yearly_summary = plot_time_series_analysis(datasets)

# Mostrar estatísticas anuais
if yearly_summary is not None:
    print(f"\nEstatísticas Anuais (primeiros/últimos 5 anos):")
    print("Primeiros 5 anos:")
    print(yearly_summary.head().round(3))
    print("\nÚltimos 5 anos:")  
    print(yearly_summary.tail().round(3))

## 6. Statistical Analysis Plots

In [ ]:
# Análises estatísticas e mapas espaciais
def create_spatial_analysis(datasets, sample_date=None):
    """Criar análises espaciais dos dados de precipitação"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análise Espacial da Precipitação', fontsize=16, fontweight='bold')
    
    # Selecionar um dataset para análise espacial (usar o mais recente disponível)
    selected_ds = None
    selected_period = None
    for period in sorted(datasets.keys(), reverse=True):
        if datasets[period] is not None:
            selected_ds = datasets[period]
            selected_period = period
            break
    
    if selected_ds is None:
        print("Nenhum dataset disponível para análise espacial!")
        return
    
    # 1. Precipitação média anual espacial
    annual_mean = selected_ds['pr'].mean(dim='time')
    
    im1 = axes[0,0].contourf(selected_ds.lon, selected_ds.lat, annual_mean, 
                            levels=20, cmap='Blues', extend='max')
    axes[0,0].set_title(f'Precipitação Média Anual ({selected_period})')
    axes[0,0].set_xlabel('Longitude')
    axes[0,0].set_ylabel('Latitude')
    cbar1 = plt.colorbar(im1, ax=axes[0,0])
    cbar1.set_label('Precipitação (mm/dia)')
    
    # 2. Desvio padrão espacial
    annual_std = selected_ds['pr'].std(dim='time')
    
    im2 = axes[0,1].contourf(selected_ds.lon, selected_ds.lat, annual_std, 
                            levels=20, cmap='Reds', extend='max')
    axes[0,1].set_title(f'Variabilidade da Precipitação ({selected_period})')
    axes[0,1].set_xlabel('Longitude')
    axes[0,1].set_ylabel('Latitude')
    cbar2 = plt.colorbar(im2, ax=axes[0,1])
    cbar2.set_label('Desvio Padrão (mm/dia)')
    
    # 3. Distribuição latitudinal
    lat_profile = annual_mean.mean(dim='lon')
    lat_std_profile = annual_std.mean(dim='lon')
    
    axes[1,0].fill_between(lat_profile.lat, lat_profile - lat_std_profile, 
                          lat_profile + lat_std_profile, alpha=0.3, label='±1 σ')
    axes[1,0].plot(lat_profile.lat, lat_profile, 'b-', linewidth=2, label='Média')
    axes[1,0].set_xlabel('Latitude')
    axes[1,0].set_ylabel('Precipitação (mm/dia)')
    axes[1,0].set_title('Perfil Latitudinal da Precipitação')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Distribuição longitudinal
    lon_profile = annual_mean.mean(dim='lat')
    lon_std_profile = annual_std.mean(dim='lat')
    
    axes[1,1].fill_between(lon_profile.lon, lon_profile - lon_std_profile, 
                          lon_profile + lon_std_profile, alpha=0.3, label='±1 σ')
    axes[1,1].plot(lon_profile.lon, lon_profile, 'g-', linewidth=2, label='Média')
    axes[1,1].set_xlabel('Longitude')
    axes[1,1].set_ylabel('Precipitação (mm/dia)')
    axes[1,1].set_title('Perfil Longitudinal da Precipitação')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return annual_mean, annual_std

def create_correlation_analysis(datasets):
    """Análise de correlação temporal e espacial"""
    
    # Selecionar dataset para análise de correlação
    selected_ds = None
    for period in sorted(datasets.keys(), reverse=True):
        if datasets[period] is not None:
            selected_ds = datasets[period] 
            break
    
    if selected_ds is None:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Análise de Correlações Espaciais e Temporais', fontsize=16, fontweight='bold')
    
    # 1. Autocorrelação temporal em pontos selecionados
    # Selecionar alguns pontos de grade para análise
    sample_lats = selected_ds.lat.values[::len(selected_ds.lat)//4][:4]
    sample_lons = selected_ds.lon.values[::len(selected_ds.lon)//4][:4]
    
    colors = ['blue', 'red', 'green', 'orange']
    for i, (lat, lon) in enumerate(zip(sample_lats, sample_lons)):
        # Encontrar ponto de grade mais próximo
        lat_idx = np.argmin(np.abs(selected_ds.lat - lat))
        lon_idx = np.argmin(np.abs(selected_ds.lon - lon))
        
        ts = selected_ds['pr'].isel(lat=lat_idx, lon=lon_idx).values
        ts = ts[~np.isnan(ts)]
        
        if len(ts) > 30:  # Garantir dados suficientes
            # Calcular autocorrelação para os primeiros 30 lags
            autocorr = [np.corrcoef(ts[:-lag], ts[lag:])[0,1] if lag > 0 
                       else 1.0 for lag in range(30)]
            
            axes[0,0].plot(range(30), autocorr, color=colors[i], 
                          label=f'Lat:{lat:.1f}, Lon:{lon:.1f}')
    
    axes[0,0].set_xlabel('Lag (dias)')
    axes[0,0].set_ylabel('Autocorrelação')
    axes[0,0].set_title('Autocorrelação Temporal')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    axes[0,0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # 2. Mapa de variância da precipitação
    precipitation_var = selected_ds['pr'].var(dim='time')
    
    im2 = axes[0,1].contourf(selected_ds.lon, selected_ds.lat, precipitation_var, 
                            levels=15, cmap='viridis')
    axes[0,1].set_title('Variância da Precipitação')
    axes[0,1].set_xlabel('Longitude')  
    axes[0,1].set_ylabel('Latitude')
    cbar2 = plt.colorbar(im2, ax=axes[0,1])
    cbar2.set_label('Variância (mm²/dia²)')
    
    # 3. Distribuição de precipitação por quantis
    pr_data = selected_ds['pr'].values.flatten()
    pr_data = pr_data[~np.isnan(pr_data)]
    
    quantiles = [0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
    quantile_values = np.percentile(pr_data, [q*100 for q in quantiles])
    
    axes[1,0].bar(range(len(quantiles)), quantile_values, alpha=0.7, color='skyblue')
    axes[1,0].set_xlabel('Quantil')
    axes[1,0].set_ylabel('Precipitação (mm)')
    axes[1,0].set_title('Distribuição por Quantis')
    axes[1,0].set_xticks(range(len(quantiles)))
    axes[1,0].set_xticklabels([f'{q:.2f}' for q in quantiles], rotation=45)
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Histograma 2D: Precipitação vs coordenadas
    # Usar amostra dos dados para visualização
    sample_size = min(10000, len(pr_data))
    sample_indices = np.random.choice(len(pr_data), sample_size, replace=False)
    
    # Obter coordenadas correspondentes
    lat_coords, lon_coords = np.meshgrid(selected_ds.lat, selected_ds.lon, indexing='ij')
    lat_flat = lat_coords.flatten()[sample_indices] 
    pr_sample = pr_data[sample_indices]
    
    axes[1,1].hexbin(lat_flat, pr_sample, gridsize=30, cmap='Blues', mincnt=1)
    axes[1,1].set_xlabel('Latitude')
    axes[1,1].set_ylabel('Precipitação (mm)')
    axes[1,1].set_title('Densidade: Precipitação vs Latitude')
    
    plt.tight_layout()
    plt.show()

# Executar análises estatísticas
print("Criando análise espacial...")
spatial_mean, spatial_std = create_spatial_analysis(datasets)

print("\nCriando análise de correlações...")
create_correlation_analysis(datasets)

## 7. Advanced Data Visualizations

In [ ]:
# Visualizações avançadas com Matplotlib e Seaborn
def create_advanced_plots(datasets):
    """Criar visualizações avançadas dos dados de precipitação"""
    
    # Combinar dados de todos os períodos
    combined_data = []
    for period, ds in sorted(datasets.items()):
        if ds is not None and 'pr' in ds.data_vars:
            # Calcular média espacial diária
            daily_mean = ds['pr'].mean(dim=['lat', 'lon'])
            times = pd.to_datetime(ds.time.values)
            
            for time, precip in zip(times, daily_mean.values):
                if not np.isnan(precip):
                    combined_data.append({
                        'date': time,
                        'precipitation': precip,
                        'period': period,
                        'year': time.year,
                        'month': time.month,
                        'day_of_year': time.timetuple().tm_yday
                    })
    
    if not combined_data:
        print("Nenhum dado disponível para visualizações avançadas!")
        return
    
    df_combined = pd.DataFrame(combined_data)
    
    # 1. Heatmap anual com seaborn
    print("Criando heatmap anual...")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Visualizações Avançadas dos Dados de Precipitação', fontsize=16, fontweight='bold')
    
    # Criar pivot para heatmap
    monthly_pivot = df_combined.groupby(['year', 'month'])['precipitation'].mean().reset_index()
    monthly_pivot = monthly_pivot.pivot(index='year', columns='month', values='precipitation')
    
    # Heatmap mensal
    sns.heatmap(monthly_pivot, cmap='Blues', cbar_kws={'label': 'Precipitação (mm/dia)'}, ax=axes[0,0])
    axes[0,0].set_title('Heatmap Mensal da Precipitação')
    axes[0,0].set_xlabel('Mês')
    axes[0,0].set_ylabel('Ano')
    
    # 2. Distribuição sazonal
    df_combined['season'] = df_combined['month'].apply(
        lambda x: 'Verão' if x in [12,1,2] else 
                 'Outono' if x in [3,4,5] else
                 'Inverno' if x in [6,7,8] else 'Primavera'
    )
    
    # Box plot sazonal por período
    season_data = []
    season_labels = []
    colors = ['lightblue', 'orange', 'lightgreen', 'pink']
    
    for season in ['Verão', 'Outono', 'Inverno', 'Primavera']:
        season_precip = df_combined[df_combined['season'] == season]['precipitation'].values
        season_data.append(season_precip)
        season_labels.append(season)
    
    bp = axes[0,1].boxplot(season_data, labels=season_labels, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    axes[0,1].set_title('Distribuição Sazonal da Precipitação')
    axes[0,1].set_ylabel('Precipitação (mm/dia)')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Análise de tendências anuais
    annual_stats = df_combined.groupby(['year', 'period'])['precipitation'].agg(['mean', 'std']).reset_index()
    
    colors = {'1961-1980': 'blue', '1981-2000': 'red', '2001-2019': 'green', '2020-2024': 'orange'}
    
    for period in annual_stats['period'].unique():
        period_data = annual_stats[annual_stats['period'] == period]
        axes[1,0].plot(period_data['year'], period_data['mean'], 'o-', 
                      color=colors.get(period, 'black'), label=period, alpha=0.8, linewidth=2)
        
        # Adicionar barras de erro
        axes[1,0].fill_between(period_data['year'], 
                              period_data['mean'] - period_data['std']/2,
                              period_data['mean'] + period_data['std']/2,
                              color=colors.get(period, 'black'), alpha=0.2)
    
    axes[1,0].set_title('Tendências Anuais por Período')
    axes[1,0].set_xlabel('Ano')
    axes[1,0].set_ylabel('Precipitação Média (mm/dia)')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Distribuição acumulada
    axes[1,1].hist(df_combined['precipitation'], bins=50, alpha=0.7, 
                   color='skyblue', edgecolor='black', density=True, label='Distribuição')
    
    # Adicionar linha da distribuição cumulativa
    ax_twin = axes[1,1].twinx()
    sorted_precip = np.sort(df_combined['precipitation'])
    cumulative = np.arange(1, len(sorted_precip) + 1) / len(sorted_precip)
    ax_twin.plot(sorted_precip, cumulative, 'r-', linewidth=2, label='Cumulativa')
    
    axes[1,1].set_xlabel('Precipitação (mm/dia)')
    axes[1,1].set_ylabel('Densidade de Probabilidade')
    ax_twin.set_ylabel('Probabilidade Cumulativa')
    axes[1,1].set_title('Distribuição e Função Cumulativa')
    axes[1,1].legend(loc='upper right')
    ax_twin.legend(loc='lower right')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 5. Análise de correlação mensal (gráfico separado)
    print("Criando matriz de correlação mensal...")
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # Criar matriz de correlação entre meses
    monthly_corr_data = df_combined.groupby(['year', 'month'])['precipitation'].mean().reset_index()
    monthly_pivot_corr = monthly_corr_data.pivot(index='year', columns='month', values='precipitation')
    correlation_matrix = monthly_pivot_corr.corr()
    
    # Criar heatmap de correlação
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
    
    ax.set_title('Correlação Entre Meses de Precipitação', fontsize=14, fontweight='bold')
    ax.set_xlabel('Mês')
    ax.set_ylabel('Mês')
    
    # Definir labels dos meses
    month_labels = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                    'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
    ax.set_xticklabels(month_labels)
    ax.set_yticklabels(month_labels)
    
    plt.tight_layout()
    plt.show()
    
    return df_combined

# Criar visualizações avançadas
print("Gerando visualizações avançadas...")
advanced_df = create_advanced_plots(datasets)

## 8. Export Plots

In [ ]:
# Função para exportar gráficos
def export_summary_plots(datasets, output_dir="../output/plots/"):
    """Criar e exportar gráficos de resumo dos dados Xavier Daily"""
    
    # Criar diretório de saída se não existir
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print(f"Exportando gráficos para: {output_path.absolute()}")
    
    # 1. Resumo estatístico geral
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Resumo Executivo - Dados Xavier Daily', fontsize=18, fontweight='bold')
    
    # Coletar estatísticas de todos os períodos
    period_stats = []
    period_labels = []
    all_data = []
    
    for period, ds in sorted(datasets.items()):
        if ds is not None and 'pr' in ds.data_vars:
            pr_data = ds['pr'].values.flatten()
            pr_data = pr_data[~np.isnan(pr_data)]
            
            if len(pr_data) > 0:
                all_data.extend(pr_data)
                period_stats.append({
                    'mean': np.mean(pr_data),
                    'median': np.median(pr_data),
                    'std': np.std(pr_data),
                    'p95': np.percentile(pr_data, 95),
                    'dry_days': np.sum(pr_data < 1.0) / len(pr_data) * 100,
                    'wet_days': np.sum(pr_data >= 1.0) / len(pr_data) * 100
                })
                period_labels.append(period)
    
    if period_stats:
        stats_df = pd.DataFrame(period_stats, index=period_labels)
        
        # Gráfico 1: Estatísticas descritivas
        x_pos = np.arange(len(period_labels))
        width = 0.2
        
        axes[0,0].bar(x_pos - width*1.5, stats_df['mean'], width, label='Média', alpha=0.8)
        axes[0,0].bar(x_pos - width*0.5, stats_df['median'], width, label='Mediana', alpha=0.8)
        axes[0,0].bar(x_pos + width*0.5, stats_df['std'], width, label='Desvio Padrão', alpha=0.8)
        axes[0,0].bar(x_pos + width*1.5, stats_df['p95'], width, label='Percentil 95', alpha=0.8)
        
        axes[0,0].set_xlabel('Período')
        axes[0,0].set_ylabel('Precipitação (mm)')
        axes[0,0].set_title('Estatísticas por Período')
        axes[0,0].set_xticks(x_pos)
        axes[0,0].set_xticklabels(period_labels, rotation=45)
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)
        
        # Gráfico 2: Distribuição geral
        axes[0,1].hist(all_data, bins=50, alpha=0.7, color='lightblue', edgecolor='black')
        axes[0,1].set_xlabel('Precipitação (mm)')
        axes[0,1].set_ylabel('Frequência')
        axes[0,1].set_title('Distribuição Geral da Precipitação')
        axes[0,1].set_xlim(0, np.percentile(all_data, 99))
        axes[0,1].grid(True, alpha=0.3)
        
        # Gráfico 3: Dias secos vs chuvosos
        axes[1,0].bar(period_labels, stats_df['dry_days'], alpha=0.8, color='orange', label='Dias Secos (<1mm)')
        axes[1,0].bar(period_labels, stats_df['wet_days'], bottom=stats_df['dry_days'], alpha=0.8, color='blue', label='Dias Chuvosos (≥1mm)')
        axes[1,0].set_xlabel('Período')
        axes[1,0].set_ylabel('Percentual de Dias (%)')
        axes[1,0].set_title('Frequência de Dias Secos vs Chuvosos')
        axes[1,0].legend()
        axes[1,0].set_xticklabels(period_labels, rotation=45)
        axes[1,0].grid(True, alpha=0.3)
        
        # Gráfico 4: Variabilidade temporal
        temporal_data = []
        temporal_labels = []
        
        for period, ds in sorted(datasets.items()):
            if ds is not None and 'pr' in ds.data_vars:
                daily_mean = ds['pr'].mean(dim=['lat', 'lon']).values
                daily_mean = daily_mean[~np.isnan(daily_mean)]
                if len(daily_mean) > 0:
                    temporal_data.append(daily_mean)
                    temporal_labels.append(period)
        
        if temporal_data:
            bp = axes[1,1].boxplot(temporal_data, labels=temporal_labels, patch_artist=True)
            colors = ['lightcoral', 'lightblue', 'lightgreen', 'lightyellow']
            for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            axes[1,1].set_xlabel('Período')
            axes[1,1].set_ylabel('Precipitação Média Diária (mm)')
            axes[1,1].set_title('Variabilidade Temporal por Período')
            axes[1,1].grid(True, alpha=0.3)
            plt.setp(axes[1,1].get_xticklabels(), rotation=45)
    
    plt.tight_layout()
    
    # Salvar em múltiplos formatos
    for fmt in ['png', 'svg', 'pdf']:
        plt.savefig(output_path / f'xavier_daily_summary.{fmt}', dpi=300, bbox_inches='tight')
    
    plt.show()
    
    # 2. Criar mapa resumo (se dados espaciais disponíveis)
    for period, ds in sorted(datasets.items(), reverse=True):
        if ds is not None and 'pr' in ds.data_vars:
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            fig.suptitle(f'Mapas de Precipitação - Período {period}', fontsize=16, fontweight='bold')
            
            # Média anual
            annual_mean = ds['pr'].mean(dim='time')
            im1 = axes[0].contourf(ds.lon, ds.lat, annual_mean, levels=20, cmap='Blues', extend='max')
            axes[0].set_title('Precipitação Média Anual')
            axes[0].set_xlabel('Longitude')
            axes[0].set_ylabel('Latitude')
            cbar1 = plt.colorbar(im1, ax=axes[0])
            cbar1.set_label('Precipitação (mm/dia)')
            
            # Desvio padrão
            annual_std = ds['pr'].std(dim='time')
            im2 = axes[1].contourf(ds.lon, ds.lat, annual_std, levels=20, cmap='Reds', extend='max')
            axes[1].set_title('Desvio Padrão da Precipitação')
            axes[1].set_xlabel('Longitude')
            axes[1].set_ylabel('Latitude')
            cbar2 = plt.colorbar(im2, ax=axes[1])
            cbar2.set_label('Desvio Padrão (mm/dia)')
            
            plt.tight_layout()
            
            # Salvar mapas
            for fmt in ['png', 'svg']:
                plt.savefig(output_path / f'spatial_analysis_{period.replace("-", "_")}.{fmt}', 
                           dpi=300, bbox_inches='tight')
            
            plt.show()
            break  # Criar mapa apenas para o período mais recente
    
    # 3. Criar relatório de metadados
    metadata_report = []
    metadata_report.append("# Relatório de Análise dos Dados Xavier Daily\\n")
    metadata_report.append(f"**Data de geração:** {pd.Timestamp.now().strftime('%d/%m/%Y %H:%M')}\\n")
    metadata_report.append(f"**Total de períodos analisados:** {len(datasets)}\\n\\n")
    
    metadata_report.append("## Resumo dos Datasets:\\n")
    for period, ds in sorted(datasets.items()):
        if ds is not None:
            metadata_report.append(f"### Período {period}")
            metadata_report.append(f"- **Arquivo:** {nc_files.get(period, 'N/A')}")
            metadata_report.append(f"- **Dimensões:** {dict(ds.dims)}")
            metadata_report.append(f"- **Período temporal:** {ds.time.min().values} a {ds.time.max().values}")
            
            if 'pr' in ds.data_vars:
                pr_data = ds['pr'].values.flatten()
                pr_data = pr_data[~np.isnan(pr_data)]
                metadata_report.append(f"- **Estatísticas da precipitação:**")
                metadata_report.append(f"  - Média: {np.mean(pr_data):.2f} mm/dia")
                metadata_report.append(f"  - Desvio padrão: {np.std(pr_data):.2f} mm/dia")
                metadata_report.append(f"  - Valores ausentes: {(1 - len(pr_data)/ds['pr'].size)*100:.1f}%")
            metadata_report.append("")
    
    # Salvar relatório
    with open(output_path / 'analysis_report.md', 'w', encoding='utf-8') as f:
        f.write('\\n'.join(metadata_report))
    
    print(f"\\nGráficos e relatório salvos em: {output_path.absolute()}")
    print("Arquivos criados:")
    for file in output_path.glob('*'):
        print(f"  - {file.name}")

# Executar exportação
export_summary_plots(datasets)

print("\\n" + "="*60)
print("ANÁLISE CONCLUÍDA!")
print("="*60)
print(f"Total de datasets processados: {len([ds for ds in datasets.values() if ds is not None])}")
print("Visualizações criadas:")
print("  ✓ Análise exploratória básica")
print("  ✓ Análise de séries temporais") 
print("  ✓ Análises estatísticas e espaciais")
print("  ✓ Visualizações avançadas")
print("  ✓ Exportação de gráficos")
print("\\nPara executar análises adicionais, utilize as funções definidas neste notebook.")
print("="*60)